# 02 — Phase 1: Baseline Signal-Only Classifier Under Domain Shift

Trains the plain signal-only ECG classifier on one PTB-XL site and
evaluates it, unmodified, on a different site — reproducing the
"domain shift hurts performance" problem from the reference paper
(proposal Section 6.2, step 1), before any fusion is introduced.

Assumes `00_colab_setup.ipynb` has already been run in this runtime.

In [1]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')

OWNER, REPO = "Sakina152", "mm-dg-healthcare"
tok = userdata.get("GH_TOKEN")
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://{tok}@github.com/{OWNER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!pip install -q wfdb transformers pytest PyYAML tqdm scikit-learn pandas matplotlib
!mkdir -p data/raw
!tar xf /content/drive/MyDrive/mm-dg-healthcare-data/ptbxl.tar -C data/raw
!ls data/raw/ptbxl | head -3

Mounted at /content/drive


SecretNotFoundError: Secret GH_TOKEN does not exist.

In [ ]:
%cd /content/mm-dg-healthcare
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Run one domain pair from the command line

Uses `configs/baseline_signal_only.yaml` as-is (site 1 → site 2). This
calls the exact same `src/training/trainer.py` used outside Colab —
nothing Colab-specific about the actual training code, only this
notebook's setup cells are Colab-specific.

In [ ]:
!python -m src.training.trainer --config configs/baseline_signal_only.yaml

## Inspect results

Every run's metrics, full epoch history, and the exact config used are
written to `experiments/results/<run_dir>/` — never just printed and
lost.

In [ ]:
import json

run_dir = "experiments/results/baseline_signal_only_site1v2"
with open(f"{run_dir}/metrics.json") as f:
    metrics = json.load(f)
metrics

In [ ]:
import pandas as pd

with open(f"{run_dir}/history.json") as f:
    history = json.load(f)

hist_df = pd.DataFrame([
    {"epoch": h["epoch"], "train_loss": h["train_loss"], "test_loss": h["test_loss"],
     "rare_class_f1": h["test_metrics"]["rare_class_f1"], "accuracy": h["test_metrics"]["accuracy"]}
    for h in history
])
hist_df.plot(x="epoch", y=["train_loss", "test_loss"], title="Loss curves")
plt.show()
hist_df.plot(x="epoch", y=["rare_class_f1", "accuracy"], title="Target-domain metrics per epoch")
plt.show()
hist_df.tail()

## Sweep every site pair (optional, matches the reference paper's full
across-trial validation table)

This trains one model per ordered site pair — with 5 PTB-XL sites that's
20 runs. On a Colab GPU each run is fast (small model, ~20 epochs), but
this cell will still take a while; run it when you actually want the full
table, not on every notebook re-run.

In [ ]:
RUN_FULL_SWEEP = False  # flip to True when you want the full table

if RUN_FULL_SWEEP:
    import yaml
    from src.data.ptbxl_dataset import load_ptbxl_metadata
    from src.data.domain_split import all_pairwise_domain_splits
    from src.utils.config import load_config, save_config
    from src.training.trainer import train as train_fn

    metadata = load_ptbxl_metadata("data/raw/ptbxl")
    splits = all_pairwise_domain_splits(metadata, "site")

    results = []
    base_cfg_dict = load_config("configs/baseline_signal_only.yaml").to_dict()

    for split in splits:
        cfg_dict = dict(base_cfg_dict)
        cfg_dict["data"] = dict(cfg_dict["data"])
        cfg_dict["data"]["train_sites"] = split.train_domains
        cfg_dict["data"]["test_sites"] = split.test_domains
        run_name = f"baseline_site{split.train_domains[0]}v{split.test_domains[0]}"
        cfg_dict["output"] = {"run_dir": f"experiments/results/{run_name}"}

        from src.utils.config import Config
        cfg = Config(cfg_dict)
        print(f"\n=== {split.summary()} ===")
        final_metrics = train_fn(cfg)
        results.append({"train_site": split.train_domains[0], "test_site": split.test_domains[0], **final_metrics})

    sweep_df = pd.DataFrame(results)
    sweep_df.to_csv("experiments/results/baseline_sweep_summary.csv", index=False)
    sweep_df

Continue to `03_rare_class_identification.ipynb` to formally identify the rare class per Definition 2.